# 05 — Surrogate Model Training

Train an ensemble MLP surrogate on **10 VLM primitives**:

**Original 7:** CL_0, CL_alpha, CM_0, CM_alpha, CD0_wing, CD0_body, Cn_beta

**v2 additions (control derivatives):** Cl_beta, CLd01 (elevon CL), Cmd01 (elevon Cm)

## Steps
1. Load dataset `eval_database_v2.json` (with consistent c_ref + control derivatives)
2. Feature augmentation (30 -> ~52)
3. Train 5-model ensemble (10 targets)
4. Per-primitive accuracy analysis
5. Save trained model to `models/surrogate_v2_ctrl`

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.surrogate.model import SurrogateModel, PRIMITIVE_TARGETS
from src.surrogate.features import augment_features
from src.surrogate.reconstruct import reconstruct_aero
from src.optimization.database import EvaluationDatabase
from src.aero.mission import MissionCondition, FeasibilityConfig

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Load Dataset

In [ ]:
db = EvaluationDatabase.load('../data/eval_database_v2.json')
X_arr, results = db.to_arrays()
print(f'Loaded {len(db)} evaluations')
print(f'X shape: {X_arr.shape}')
n_success = sum(1 for r in results if r.get('success', True))
n_ctrl = sum(1 for r in results if 'Cmd01' in r)
n_cref = sum(1 for r in results if 'c_ref' in r)
print(f'Successful VLM: {n_success}/{len(results)}')
print(f'With control derivatives: {n_ctrl}/{len(results)}')
print(f'With consistent c_ref: {n_cref}/{len(results)}')

# Quick stats on new targets
cmd01_vals = np.array([r['Cmd01'] for r in results if 'Cmd01' in r])
cld01_vals = np.array([r['CLd01'] for r in results if 'CLd01' in r])
clb_vals = np.array([r.get('Cl_beta', 0) for r in results if 'Cl_beta' in r])
print(f'\nNew target stats:')
print(f'  Cmd01:   [{cmd01_vals.min():.5f}, {cmd01_vals.max():.5f}] mean={cmd01_vals.mean():.5f}')
print(f'  CLd01:   [{cld01_vals.min():.5f}, {cld01_vals.max():.5f}] mean={cld01_vals.mean():.5f}')
print(f'  Cl_beta: [{clb_vals.min():.5f}, {clb_vals.max():.5f}] mean={clb_vals.mean():.5f}')

## 2. Train Surrogate

In [ ]:
# Train 10-target surrogate (7 original + 3 control derivatives)
surrogate = SurrogateModel(n_ensemble=5, _n_targets=10)
success = surrogate.fit(X_arr, results, min_samples=30)

if success:
    print(f'Surrogate trained successfully!')
    print(f'Architecture: MLP {surrogate.architecture_str}')
    print(f'Targets ({surrogate._n_targets}): {surrogate.target_keys}')
else:
    print('Training failed -- not enough valid samples')

## 3. Accuracy Analysis — Parity Plots

In [ ]:
# Predict on CLEAN dataset only (filtered by _TARGET_CLIP)
if surrogate.is_trained:
    from src.surrogate.model import _TARGET_CLIP
    
    targets = surrogate.target_keys
    n_targets = len(targets)
    
    # Re-extract targets and apply same filter as fit()
    Y_rows, valid_idx = [], []
    for i, r in enumerate(results):
        if not r.get('success', True):
            continue
        alpha_eq_rad = np.radians(r.get('alpha_eq', 0.0))
        cl_alpha = r.get('CL_alpha', 0.0)
        cm_alpha = r.get('CM_alpha', 0.0)
        cl_0 = r.get('CL_0', None)
        if cl_0 is None:
            cl_0 = r.get('CL_required', 0.0) - cl_alpha * alpha_eq_rad
        cm_0 = r.get('CM_0', None)
        if cm_0 is None:
            cm_0 = r.get('CM', 0.0) - cm_alpha * alpha_eq_rad
        cd0_wing = r.get('CD0_wing', r.get('CD0', 0.008) * 0.6)
        cd0_body = r.get('CD0_body', r.get('CD0', 0.008) * 0.4)
        cn_beta = r.get('Cn_beta', 0.0)
        
        row = [cl_0, cl_alpha, cm_0, cm_alpha, cd0_wing, cd0_body, cn_beta]
        
        # v2 control derivatives
        if n_targets == 10:
            cl_beta = r.get('Cl_beta', r.get('Cl_beta_ctrl', None))
            cl_d01 = r.get('CLd01', None)
            cmd01 = r.get('Cmd01', None)
            if cl_d01 is None or cmd01 is None:
                continue
            row.extend([cl_beta or 0.0, cl_d01, cmd01])
        
        Y_rows.append(row)
        valid_idx.append(i)
    
    Y_all = np.array(Y_rows)
    X_valid = X_arr[valid_idx]
    
    # Apply _TARGET_CLIP filter
    mask = np.ones(len(Y_all), dtype=bool)
    for j, key in enumerate(targets):
        if key in _TARGET_CLIP:
            lo, hi = _TARGET_CLIP[key]
            mask &= (Y_all[:, j] >= lo) & (Y_all[:, j] <= hi)
    
    X_clean = X_valid[mask]
    Y_clean = Y_all[mask]
    print(f'Evaluating on {mask.sum()} clean samples (out of {len(Y_all)} successful)')
    
    preds_clean = surrogate.predict(X_clean)
    
    n_cols = 4
    n_rows = (n_targets + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
    axes = axes.flat
    
    for i, key in enumerate(targets):
        ax = axes[i]
        y_true = Y_clean[:, i]
        y_pred = preds_clean[key]
        
        # R²
        ss_res = np.sum((y_true - y_pred)**2)
        ss_tot = np.sum((y_true - np.mean(y_true))**2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        
        # NRMSE
        rmse = np.sqrt(np.mean((y_true - y_pred)**2))
        rng = y_true.max() - y_true.min()
        nrmse = rmse / rng * 100 if rng > 1e-8 else 0
        
        ax.scatter(y_true, y_pred, s=2, alpha=0.3)
        lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, 'r--', lw=1)
        ax.set_xlabel('True')
        ax.set_ylabel('Predicted')
        ax.set_title(f'{key}\nR²={r2:.4f}  NRMSE={nrmse:.1f}%')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
    
    # Hide unused axes
    for j in range(n_targets, len(axes)):
        axes[j].set_visible(False)
    
    plt.tight_layout()
    plt.show()

## 4. Save Model

In [ ]:
if surrogate.is_trained:
    save_path = '../models/surrogate_v2_ctrl'
    surrogate.save(save_path)
    print(f'Surrogate saved to {save_path}/')
    print(f'  Targets: {surrogate.target_keys}')
    print(f'  Ensemble: {surrogate.n_ensemble} models')
    print(f'  Architecture: {surrogate.architecture_str}')